# 01 · The ML workflow and the train/test split

After this notebook you can frame a problem as *features → label*, split data into train / validation / test without leaking, check a model against a dumb baseline, and drive any scikit-learn model with `fit` / `predict` / `score`.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb), [pandas essentials](../04_numpy_pandas_pytorch/02_pandas_essentials.ipynb)

## Why this matters in interviews

- "How would you build a classifier for X?" is the warm-up in almost every ML round. A clean *frame → split → baseline → model → test once* answer is what they listen for.
- Every GenAI evaluation question is this notebook in disguise: an **eval set is a test set**, **benchmark contamination is leakage**, and "we beat the keyword rule" is **beating a baseline**.
- Leakage is the most common reason a model that looked great offline disappoints in production. Interviewers love asking you to spot it.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo11` | What is the difference between AI, machine learning, deep learning and generative AI? |
| `fo01` | What is generative AI, and how is it different from the machine learning that came before it? |
| `dt22` | How do you sample data for evaluation without biasing the result? |
| `ev05` | How do you build an eval set from scratch? |
| `ev20` | What is benchmark contamination and why should you care about your own evals? |
| `dt14` | How do you build a training dataset from production logs? |
| `tn25` | How would you build a fine-tuning dataset from production traffic? |

**Also asked in classical-ML rounds:** why three splits and not two? what does `stratify` do? name four kinds of leakage. what is your baseline?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import NotFittedError
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, "#eda100", "#4a3aa7"])})

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

## 1. Framing: features, labels and the kinds of learning

**In plain English:** a *row* is one example (a patient, an email, a customer). The columns you will know *at prediction time* are the **features** `X`. The thing you want to predict is the **label** `y`. If you have labels for past rows, it is **supervised** learning; if you only have `X` and want structure, it is **unsupervised**.

| Kind | You have | Output | Examples | Where in this section |
|---|---|---|---|---|
| Supervised: classification | `X` + a category `y` | a class (and a probability) | spam/ham, fraud, "is this prompt unsafe?" | 03, 06, 07 |
| Supervised: regression | `X` + a number `y` | a number | price, latency, days to resolve a ticket | 02, 06 |
| Unsupervised | `X` only | groups, directions, outliers | customer segments, topic clusters, embedding maps | 08 |
| Self-supervised | raw data, the label is cut from the data itself | e.g. the next token | LLM pretraining | [07_genai_foundations](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb) |

**Where GenAI fits (`fo01`, `fo11`).** AI ⊃ machine learning ⊃ deep learning ⊃ generative models ⊃ LLMs. Classical ML is mostly *discriminative*: it learns P(y | x) over a fixed label set, e.g. {malignant, benign}. An LLM is trained on the same supervised recipe, except the "label" is the next token and the classes are a vocabulary of 50k to 200k tokens; generating text is running that classifier again and again. Gradient-boosted trees still win on tabular data, which is why this section exists.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target                  # X: DataFrame of features, y: Series of labels
print(f"X shape: {X.shape}   (rows = tumours, columns = measured features)")
names = data.target_names.tolist()
print("label meaning:", dict(enumerate(names)))
print("label counts: ", {names[k]: int(v) for k, v in y.value_counts().sort_index().items()})
X.iloc[:3, :5]

Every supervised project follows the same loop. The rest of this notebook walks through it; later notebooks go deep on each box.

In [ ]:
draw_pipeline(["Frame\nX -> y", "Split\ntrain/val/test", "Baseline\n(dummy)", "Fit on\ntrain",
               "Choose on\nvalidation", "Test\nonce", "Ship +\nmonitor"], title="The supervised ML workflow")

## 2. Why three splits: train, validation, test

**In plain English:** the training set is the textbook. The validation set is the practice exams you use to decide *how* to study (which model, which settings). The test set is the final exam, and you sit it **once**. If you pick your model by looking at the final exam, your grade stops meaning anything.

Why? *Selecting* the best of many options on one set rewards luck. Below: 200 "models" that are pure noise (random linear rules on random features, labels are coin flips, so every model's true accuracy is 50%). We pick the best one on a 100-row validation set, then score **the same model** on fresh data.

In [ ]:
X_noise = rng.normal(size=(10_100, 20))
y_noise = rng.integers(0, 2, size=10_100)             # labels are coin flips: nothing to learn
W = rng.normal(size=(20, 200))                        # 200 random linear "models"
preds = (X_noise @ W > 0).astype(int)                 # shape (rows, models)

val, fresh = slice(0, 100), slice(100, None)
val_acc = (preds[val] == y_noise[val, None]).mean(axis=0)
best = val_acc.argmax()
fresh_acc = (preds[fresh, best] == y_noise[fresh]).mean()
print(f"best of 200 on the 100-row validation set: {val_acc[best]:.2f}")
print(f"same model on 10,000 fresh rows:           {fresh_acc:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(val_acc, bins=20, color=BLUE, edgecolor="white")
ax.axvline(0.5, color=GREY, ls="--", label="true accuracy of every model (0.5)")
ax.axvline(val_acc[best], color=ORANGE, lw=2, label=f"the one we 'picked' ({val_acc[best]:.2f})")
ax.set(title="Validation accuracy of 200 useless models", xlabel="accuracy on 100 validation rows", ylabel="models")
ax.legend(); plt.show()
assert val_acc[best] >= 0.6 and abs(fresh_acc - 0.5) < 0.03   # the "winner" was luck

The winner's score is inflated by the act of choosing, and the inflation disappears on data that played no part in the choice. That is the whole argument for a **test set you touch once**: validation absorbs the optimism of model selection; the test set gives an unbiased final number. The same trap in GenAI: tuning a prompt against the same 50 eval questions for six weeks (`ev26`, in [notebook 05](05_overfitting_regularization_cross_validation.ipynb)).

## 3. `train_test_split`, `stratify` and a three-way split

`train_test_split` shuffles and cuts. `stratify=y` makes every split keep the same class mix as the full data. `random_state` makes it reproducible. There is no built-in three-way split: call it twice.

In [ ]:
def three_way_split(X, y, val=0.2, test=0.2, seed=0):
    """Stratified train / validation / test split (default 60 / 20 / 20)."""
    X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=test, stratify=y, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(
        X_rest, y_rest, test_size=val / (1 - test), stratify=y_rest, random_state=seed)
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(X, y)
for name, part in [("all", y), ("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:>5}: {len(part):3d} rows, benign share = {part.mean():.3f}")
assert len(y_train) + len(y_val) + len(y_test) == len(y)
assert all(abs(part.mean() - y.mean()) < 0.01 for part in (y_train, y_val, y_test))

With 569 rows and a 37/63 class mix, even a random split would land close. `stratify` earns its keep on **small or imbalanced** data. Here: 200 rows with 10% positives, split 300 different ways, with and without `stratify`.

In [ ]:
Xs, ys = make_classification(n_samples=200, weights=[0.9], flip_y=0, random_state=0)   # 180 negatives, 20 positives
rates = {"random split": [], "stratified split": []}
for seed in range(300):
    *_, yt = train_test_split(Xs, ys, test_size=0.2, random_state=seed)
    *_, yt_s = train_test_split(Xs, ys, test_size=0.2, random_state=seed, stratify=ys)
    rates["random split"].append(yt.mean()); rates["stratified split"].append(yt_s.mean())

fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.arange(0, 0.30, 0.025)
ax.hist(rates["random split"], bins=bins, color=BLUE, alpha=0.8, label="random split")
ax.hist(rates["stratified split"], bins=bins, color=ORANGE, alpha=0.8, label="stratified split")
ax.set(title="Positive rate in a 40-row test set, over 300 random seeds (true rate = 0.10)",
       xlabel="share of positives in the test set", ylabel="seeds"); ax.legend(); plt.show()
for k, v in rates.items():
    print(f"{k:>16}: min {min(v):.3f}  max {max(v):.3f}  std {np.std(v):.3f}")
assert np.std(rates["stratified split"]) < np.std(rates["random split"]) / 5

Without `stratify`, some test sets got no positives at all and others more than double the true rate, so any metric computed on them swings for no reason. With `stratify`, every seed gives exactly 4 positives in 40. The eval-set version of this (`dt22`, `ev05`) is **stratified sampling across intents, languages and tenants**, so rare-but-important cases are guaranteed to be in the set.

## 4. Data leakage: four ways to fool yourself

**In plain English:** leakage is when information that will *not* be available at prediction time sneaks into training or evaluation. The model looks brilliant offline and ordinary in production.

| Leak | What happens | Fix |
|---|---|---|
| Preprocessing fit on all data | test-set statistics (means, vocabulary, selected features) shape training | fit every transformer on train only; use a `Pipeline` ([09](09_feature_engineering_and_pipelines.ipynb)) |
| Target leak | a feature is computed from, or recorded after, the label | ask of every column: *would I have this value at prediction time?* |
| Duplicates across the split | the same or near-same example sits in train and test, so memorising looks like skill | dedupe; split by group (user, document, conversation) |
| Time leak | a random split lets the model train on the future | train on the past, test on the most recent slice |

### 4a. Scaling fit on all the data vs on train only

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
knn = KNeighborsClassifier(n_neighbors=5)

scaler_all = StandardScaler().fit(X)            # WRONG: has seen the test rows
scaler_tr = StandardScaler().fit(X_tr)          # RIGHT: train rows only
acc_wrong = clone(knn).fit(scaler_all.transform(X_tr), y_tr).score(scaler_all.transform(X_te), y_te)
acc_right = clone(knn).fit(scaler_tr.transform(X_tr), y_tr).score(scaler_tr.transform(X_te), y_te)

shift = np.abs(scaler_all.mean_ - scaler_tr.mean_) / scaler_tr.scale_
print(f"learned means differ by up to {shift.max():.3f} standard deviations")
print(f"kNN accuracy, scaler fit on ALL rows : {acc_wrong:.3f}")
print(f"kNN accuracy, scaler fit on TRAIN    : {acc_right:.3f}")

Honest result: for plain scaling the scores are identical here. A mean over 426 rows is nearly the same number as a mean over 569. Do not conclude that leakage is harmless. The *bug* is that the offline pipeline did something production cannot do (at serving time you get one row, not the whole test set). How much it costs depends on how much the step learns: a mean is one number per feature, while a feature selector or a target encoder learns from the labels themselves. [Notebook 09](09_feature_engineering_and_pipelines.ipynb) shows a feature selector turning pure noise into an impressive cross-validation score. The habit that prevents all of it: `fit` on train, `transform` on everything else.

### 4b. A target-leaking feature

A hospital table often contains columns filled in *after* the diagnosis, such as a treatment code. Add one and watch the model become "perfect".

In [ ]:
X_leak = X.copy()
X_leak["treatment_code"] = (y == 0).astype(int)          # recorded after diagnosis: 1 = oncology referral
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y, test_size=0.25, stratify=y, random_state=0)

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
acc_honest = clone(tree).fit(Xl_tr.drop(columns="treatment_code"), yl_tr).score(Xl_te.drop(columns="treatment_code"), yl_te)
leaky = clone(tree).fit(Xl_tr, yl_tr)
acc_leaky = leaky.score(Xl_te, yl_te)
top = pd.Series(leaky.feature_importances_, index=X_leak.columns).sort_values(ascending=False)
print(f"accuracy without the leak: {acc_honest:.3f}")
print(f"accuracy with the leak   : {acc_leaky:.3f}")
print("top feature importances:", top.head(3).round(3).to_dict())
assert acc_leaky == 1.0 and top.index[0] == "treatment_code" and top.iloc[0] > 0.9

Two red flags to say out loud in an interview: **a score that is too good to be true**, and **one feature that explains nearly everything**. Either one means "go and check when that column gets filled in".

### 4c. Duplicates across the split (= benchmark contamination, `ev20`)

Production data repeats itself: the same ticket logged three times, the same document chunked with overlap. Below, every example appears 3 times with tiny noise. A 1-nearest-neighbour model (a pure memoriser) is scored with a random split and with a *group* split that keeps all copies of an example on the same side.

In [ ]:
Xd, yd = make_classification(n_samples=300, n_features=8, n_informative=3, n_redundant=0,
                             flip_y=0.15, class_sep=0.7, random_state=1)
copies = 3
X_dup = np.repeat(Xd, copies, axis=0) + rng.normal(scale=0.01, size=(len(Xd) * copies, Xd.shape[1]))
y_dup = np.repeat(yd, copies)
groups = np.repeat(np.arange(len(Xd)), copies)           # which original example each row came from

one_nn = KNeighborsClassifier(n_neighbors=1)
Xa, Xb, ya, yb = train_test_split(X_dup, y_dup, test_size=0.25, random_state=0)
acc_random_split = clone(one_nn).fit(Xa, ya).score(Xb, yb)

tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(X_dup, y_dup, groups))
assert set(groups[tr_idx]).isdisjoint(groups[te_idx])    # no example on both sides
acc_group_split = clone(one_nn).fit(X_dup[tr_idx], y_dup[tr_idx]).score(X_dup[te_idx], y_dup[te_idx])
print(f"1-NN accuracy, random split (copies leak): {acc_random_split:.3f}")
print(f"1-NN accuracy, group split (honest)      : {acc_group_split:.3f}")
assert acc_random_split > acc_group_split + 0.15

The random split rewards memorisation because most test rows have a twin in train. This is exactly benchmark contamination: if the eval questions were in the training data, the score measures recall, not skill. For LLM eval sets and fine-tuning data, **dedupe first, then split by conversation, user or document**.

### 4d. Time leak: split by time, not at random (`dt14`, `tn25`)

The world drifts. Below, the true decision rule rotates slowly over 2,000 "days". A random split mixes future days into training; a time split trains on the oldest 80% and tests on the newest 20%, which is what deployment actually looks like.

In [ ]:
n = 2000
t = np.linspace(0, 1, n)                                  # time, oldest -> newest
Xt = rng.normal(size=(n, 2))
angle = np.deg2rad(90 * t)                                # the true boundary rotates 90 degrees over the period
yt = ((Xt[:, 0] * np.cos(angle) + Xt[:, 1] * np.sin(angle)) > 0).astype(int)

Xa, Xb, ya, yb = train_test_split(Xt, yt, test_size=0.2, random_state=0)
acc_random = LogisticRegression().fit(Xa, ya).score(Xb, yb)
cut = int(0.8 * n)
acc_time = LogisticRegression().fit(Xt[:cut], yt[:cut]).score(Xt[cut:], yt[cut:])
print(f"random split accuracy: {acc_random:.3f}   (optimistic: it has seen the future)")
print(f"time split accuracy  : {acc_time:.3f}   (what you will actually get next month)")
assert acc_random > acc_time + 0.05

## 5. Baselines: what does "doing nothing" score?

**In plain English:** before you celebrate a number, find out what a model that ignores the input would score. `DummyClassifier` is that model.

| `strategy=` | Predicts | Use it to answer |
|---|---|---|
| `"most_frequent"` / `"prior"` | always the majority class | "is my accuracy above the class ratio?" |
| `"stratified"` | random labels with the training class mix | "is my model better than informed guessing?" |
| `"uniform"` | random labels, all classes equally likely | chance level |
| `DummyRegressor(strategy="mean")` | the training mean | the R² = 0 reference for regression |

GenAI baselines play the same role: a keyword rule, BM25-only retrieval, the previous prompt version, "always route to the small model".

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"dummy accuracy on breast cancer: {dummy.score(X_test, y_test):.3f}  (benign share of test = {y_test.mean():.3f})")

Xi, yi = make_classification(n_samples=2000, weights=[0.95], flip_y=0, random_state=0)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.25, stratify=yi, random_state=0)
dummy_i = DummyClassifier(strategy="most_frequent").fit(Xi_tr, yi_tr)
print(f"dummy accuracy on 95/5 data    : {dummy_i.score(Xi_te, yi_te):.3f}  <- 'high' accuracy, zero skill")
assert np.isclose(dummy.score(X_test, y_test), y_test.mean())

A 95%-accurate model on 95/5 data may be doing nothing at all. [Notebook 04](04_metrics_confusion_matrix_roc_pr.ipynb) fixes this with precision, recall and PR curves.

## 6. The scikit-learn estimator API

Everything in scikit-learn speaks one small interface, so swapping models is a one-line change.

| Method / attribute | Who has it | What it does |
|---|---|---|
| `fit(X, y)` | every estimator | learns from training data, returns `self` |
| `predict(X)` | classifiers, regressors, clusterers | labels or numbers |
| `predict_proba(X)` | most classifiers | one probability per class; rows sum to 1 |
| `decision_function(X)` | linear models, SVMs | raw scores (logits, margins), not probabilities |
| `score(X, y)` | classifiers (accuracy), regressors (R²) | the default metric |
| `transform(X)`, `fit_transform(X)` | scalers, encoders, PCA, vectorisers | new features |
| trailing underscore: `coef_`, `classes_` | fitted estimators | things learned from data |
| `get_params`, `set_params`, `clone` | every estimator | hyperparameters; `clone` = same settings, no learned weights |

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
proba = model.predict_proba(X_test)
print("classes_:", model.classes_, "   coef_ shape:", model[-1].coef_.shape)
print("first 3 rows of predict_proba:\n", proba[:3].round(3))
print(f"score() = {model.score(X_test, y_test):.3f}")

assert np.allclose(proba.sum(axis=1), 1)                                    # probabilities per row sum to 1
assert (model.predict(X_test) == model.classes_[proba.argmax(axis=1)]).all()  # predict = most probable class
assert np.isclose(model.score(X_test, y_test), accuracy_score(y_test, model.predict(X_test)))
try:
    clone(model).predict(X_test[:1])
except NotFittedError:
    print("clone(model) has the same settings but no learned weights -> NotFittedError")

## 7. End to end: a mini workflow on `load_breast_cancer`

Frame (done), split 60/20/20 with `stratify`, check the class balance, compare candidates to the baseline **on validation**, refit the winner on train + validation, and score the test set once.

In [ ]:
splits = {"train": y_train, "validation": y_val, "test": y_test}
fig, ax = plt.subplots(figsize=(8, 3.4))
xpos = np.arange(len(splits))
for offset, cls, color in [(-0.2, 0, ORANGE), (0.2, 1, BLUE)]:
    counts = [int((part == cls).sum()) for part in splits.values()]
    shares = [c / len(part) for c, part in zip(counts, splits.values())]
    bars = ax.bar(xpos + offset, shares, width=0.38, color=color, label=data.target_names[cls])
    ax.bar_label(bars, labels=[f"{c}" for c in counts], padding=2, fontsize=8)
ax.set_xticks(xpos, [f"{k}\n({len(v)} rows)" for k, v in splits.items()])
ax.set(title="Class balance per split (bar labels = row counts)", ylabel="share of rows", ylim=(0, 0.8))
ax.legend(); plt.show()

In [ ]:
candidates = {
    "dummy (majority)": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=0),
}
val_scores = {name: clone(m).fit(X_train, y_train).score(X_val, y_val) for name, m in candidates.items()}
for name, s in val_scores.items():
    print(f"{name:>24}: validation accuracy {s:.3f}")

fig, ax = plt.subplots(figsize=(8, 2.8))
names = [n for n in val_scores if not n.startswith("dummy")]
ax.barh(names, [val_scores[n] for n in names], color=BLUE)
ax.axvline(val_scores["dummy (majority)"], color=GREY, ls="--", label="dummy baseline")
ax.set(title="Choose on validation, never on test", xlabel="validation accuracy", xlim=(0.5, 1.0))
ax.legend(loc="lower right"); plt.show()

In [ ]:
best_name = max(names, key=val_scores.get)
final = clone(candidates[best_name]).fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
test_pred = final.predict(X_test)                      # the one and only look at the test set
test_acc = accuracy_score(y_test, test_pred)
missed = int(((y_test == 0) & (test_pred == 1)).sum())  # malignant tumours called benign
print(f"chosen on validation: {best_name}")
print(f"test accuracy: {test_acc:.3f}   (dummy baseline on the same test set: {dummy.score(X_test, y_test):.3f})")
print(f"malignant tumours missed on test: {missed} of {int((y_test == 0).sum())}")
assert test_acc > dummy.score(X_test, y_test) + 0.25

Accuracy is the headline, but for a cancer screen the number that matters is the missed malignant cases (false negatives, i.e. recall on the malignant class). Choosing *which* metric is [notebook 04](04_metrics_confusion_matrix_roc_pr.ipynb).

## Try it yourself

**1.** Make a stratified 70 / 15 / 15 split of `load_wine` (3 classes). Print each split's class proportions and assert they are within 3 percentage points of the full data.

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.datasets import load_wine
Xw, yw = load_wine(return_X_y=True)
Xw_train, Xw_val, Xw_test, yw_train, yw_val, yw_test = three_way_split(Xw, yw, val=0.15, test=0.15, seed=1)
full = np.bincount(yw) / len(yw)
for name, part in [("train", yw_train), ("val", yw_val), ("test", yw_test)]:
    share = np.bincount(part, minlength=3) / len(part)
    print(f"{name:>5}: {len(part):3d} rows, class shares {share.round(3)}")
    assert np.abs(share - full).max() < 0.03
print(f" full: {len(yw):3d} rows, class shares {full.round(3)}")

**2.** Leak hunt. The DataFrame below hides one leaking column among the wine features. Find it *without* reading the code that made it: score every column on its own with a depth-2 tree and 5-fold cross-validation, and look for the one that is suspiciously good.

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.model_selection import cross_val_score
wine = load_wine(as_frame=True)
df = wine.data.copy()
df["batch_label"] = wine.target * 10 + rng.integers(0, 3, size=len(df))   # the leak: derived from the target
single = {col: cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), df[[col]], wine.target, cv=5).mean()
          for col in df.columns}
ranked = pd.Series(single).sort_values(ascending=False)
print(ranked.head(4).round(3))
assert ranked.index[0] == "batch_label" and ranked.iloc[0] > 0.95

**3.** On 97 / 3 data, compare a `DummyClassifier` with a `LogisticRegression`: accuracy *and* recall on the rare class (`sklearn.metrics.recall_score`). Which number tells you the dummy is useless?

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.metrics import recall_score
Xr, yr = make_classification(n_samples=3000, weights=[0.97], flip_y=0, class_sep=1.5, random_state=0)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, stratify=yr, random_state=0)
for name, m in [("dummy", DummyClassifier(strategy="most_frequent")), ("logistic", LogisticRegression(max_iter=1000))]:
    p = m.fit(Xr_tr, yr_tr).predict(Xr_te)
    print(f"{name:>8}: accuracy {accuracy_score(yr_te, p):.3f}   rare-class recall {recall_score(yr_te, p):.3f}")
# accuracy barely separates them; recall on the rare class does (the dummy's is 0)

## Say it in an interview

- **30-second answer:** "I frame the problem as features I will have at prediction time mapped to a label. I split into train, validation and test: stratified, or by group or time if rows are related. I set a dumb baseline, fit on train, choose models and settings on validation (or cross-validation when data is small), and touch the test set once. Then I monitor, because production is the real test set."
- **Why three splits:** choosing is fitting. In the demo above, the best of 200 useless models scored 0.61 on 100 validation rows and 0.50 on fresh data. The test set exists to be untouched by choices.
- **Leakage checklist:** preprocessing fit on all data; features recorded after the label; duplicates or related rows across the split (the contamination problem); random splits on time-ordered data. Symptoms: too good to be true, one dominant feature, a large offline-to-online gap.
- **Baseline first:** `DummyClassifier` gives the class ratio for free. On 95/5 data, 95% accuracy is zero skill.
- **GenAI translation:** test set = frozen golden eval set; leakage = benchmark contamination; group split = split by conversation, user or document; time split = hold out the most recent traffic; baseline = keyword rule or previous prompt.
- **Follow-ups they ask:** "What if you only have 300 rows?" (cross-validation, [05](05_overfitting_regularization_cross_validation.ipynb)). "How do you stop preprocessing leakage?" (a `Pipeline`, [09](09_feature_engineering_and_pipelines.ipynb)). "The test score is 99.8%, what do you do?" (hunt for leakage before celebrating).

## Next

- [02 · Linear regression and gradient descent](02_linear_regression_and_gradient_descent.ipynb): the first model, built from scratch.
- [04 · Metrics: confusion matrix, ROC, PR](04_metrics_confusion_matrix_roc_pr.ipynb): why accuracy lies on imbalanced data.
- [05 · Overfitting, regularisation, cross-validation](05_overfitting_regularization_cross_validation.ipynb): validation done properly.
- [09 · Feature engineering and pipelines](09_feature_engineering_and_pipelines.ipynb): leakage-proof preprocessing.
- [ML model end to end (coding round)](../17_interview_coding_rounds/06_ml_model_end_to_end.ipynb) and [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb): the same workflow applied to GenAI.
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapters 1, 2 and 12), [interview bank](../../ai_interview_prep/index.html).